# Working with Other Hockey Leagues

This notebook demonstrates how to scrape data from QMJHL, OHL, WHL, and AHL.

All of these leagues use the HockeyTech platform, so they share a consistent API.

## Supported Leagues

- **QMJHL** - Quebec Major Junior Hockey League
- **OHL** - Ontario Hockey League
- **WHL** - Western Hockey League
- **AHL** - American Hockey League

In [ ]:
from scrapernhl import scrape, HockeyScraper
import pandas as pd

## QMJHL Examples

The Quebec Major Junior Hockey League features teams from Quebec, New Brunswick, and Prince Edward Island.

In [ ]:
# Create a QMJHL scraper
qmjhl = HockeyScraper('qmjhl')

# Get top skaters for current season (season 90 = 2024-25)
stats = qmjhl.player_stats(season=90, position='skaters', limit=20)

print(f"Top 20 QMJHL skaters:")
display(stats[['player_name', 'team', 'games_played', 'goals', 'assists', 'points']].head(20))

In [ ]:
# Get goalie stats
goalies = qmjhl.player_stats(season=90, position='goalies', limit=15)

print("Top 15 QMJHL goalies:")
display(goalies[['player_name', 'team', 'games_played', 'wins', 'save_pct', 'gaa']].head(15))

In [ ]:
# Get play-by-play for a game
# Note: Replace with an actual game ID from the current season
game_id = 31171  # Example game ID

try:
    pbp = qmjhl.play_by_play(game_id=game_id, nhlify=True)
    print(f"Total events: {len(pbp)}")
    display(pbp.head(15))
except Exception as e:
    print(f"Error getting play-by-play: {e}")
    print("Try a different game_id from the current season")

## OHL Examples

The Ontario Hockey League is one of the three major junior hockey leagues in Canada.

In [ ]:
# Create an OHL scraper
ohl = HockeyScraper('ohl')

# Get player stats (season 83 = 2024-25)
stats = ohl.player_stats(season=83, position='skaters', limit=25)

print("Top 25 OHL skaters:")
display(stats[['player_name', 'team', 'goals', 'assists', 'points']].head(25))

In [ ]:
# Get standings
standings = ohl.standings(season=83)

print("OHL Standings:")
display(standings[['team_name', 'division', 'wins', 'losses', 'points']].head())

In [ ]:
# Get schedule for all teams
schedule = ohl.schedule(season=83, team_id=-1)  # -1 = all teams

print(f"Total games in schedule: {len(schedule)}")
display(schedule[['date', 'home_team', 'away_team', 'status']].head(10))

## WHL Examples

The Western Hockey League covers teams in Western Canada and the Northwestern United States.

In [ ]:
# Functional API example for WHL
stats = scrape('whl', 'stats', season=75, position='skaters', limit=20)

print("Top 20 WHL skaters:")
display(stats[['player_name', 'team', 'goals', 'assists', 'points']].head(20))

In [ ]:
# Get WHL standings
standings = scrape('whl', 'standings', season=75)

print("WHL Standings:")
display(standings[['team_name', 'wins', 'losses', 'points']].head())

## AHL Examples

The American Hockey League is the primary development league for the NHL.

In [ ]:
# Create an AHL scraper
ahl = HockeyScraper('ahl')

# Get player stats (season 90 = 2024-25)
stats = ahl.player_stats(season=90, position='skaters', limit=30)

print("Top 30 AHL skaters:")
display(stats[['player_name', 'team', 'goals', 'assists', 'points']].head(30))

In [ ]:
# Get AHL goalie stats
goalies = ahl.player_stats(season=90, position='goalies', limit=20)

print("Top 20 AHL goalies:")
display(goalies[['player_name', 'team', 'games_played', 'wins', 'save_pct', 'gaa']].head(20))

## Comparing Across Leagues

Since all leagues return pandas DataFrames with similar structures, it's easy to compare across leagues.

In [ ]:
# Get top scorer from each league
leagues = ['qmjhl', 'ohl', 'whl']
seasons = {'qmjhl': 90, 'ohl': 83, 'whl': 75}

top_scorers = []

for league in leagues:
    stats = scrape(league, 'stats', season=seasons[league], position='skaters', limit=1)
    if len(stats) > 0:
        top_scorer = stats.iloc[0]
        top_scorers.append({
            'league': league.upper(),
            'player': top_scorer.get('player_name', 'N/A'),
            'team': top_scorer.get('team', 'N/A'),
            'points': top_scorer.get('points', 0)
        })

comparison = pd.DataFrame(top_scorers)
print("Top scorer in each league:")
display(comparison)

## Finding Season IDs

Each league uses different season IDs. Here are the current ones (2024-25 season):

- **QMJHL**: 90
- **OHL**: 83
- **WHL**: 75
- **AHL**: 90

To find the current season ID, you can look at the league's website or use trial and error.

## Team IDs

Each team has a unique ID. To get all teams and their IDs:
- Use `team_id=-1` in schedule queries to see all teams
- Check the league's website
- Look at roster or stats data which includes team information

In [ ]:
# Example: Get all teams from a schedule query
schedule = scrape('ohl', 'schedule', season=83, team_id=-1)

if 'home_team' in schedule.columns and 'away_team' in schedule.columns:
    teams = pd.concat([schedule['home_team'], schedule['away_team']]).unique()
    print(f"OHL Teams ({len(teams)}):")
    for team in sorted(teams):
        print(f"  - {team}")

## Data Transformation: nhlify

The `nhlify=True` parameter in play-by-play queries applies NHL-style transformations:
- Merges shot and goal events
- Standardizes event types
- Cleans up data inconsistencies

In [ ]:
# Example: Compare with and without nhlify
game_id = 31171  # Replace with valid game ID

try:
    # Without nhlify
    pbp_raw = scrape('qmjhl', 'pbp', game_id=game_id, nhlify=False)
    
    # With nhlify
    pbp_clean = scrape('qmjhl', 'pbp', game_id=game_id, nhlify=True)
    
    print(f"Raw events: {len(pbp_raw)}")
    print(f"Cleaned events: {len(pbp_clean)}")
except Exception as e:
    print(f"Error: {e}")

## Summary

This notebook covered:
- QMJHL, OHL, WHL, and AHL data scraping
- Player stats (skaters and goalies)
- Standings and schedules
- Play-by-play data
- Comparing across leagues
- Data transformation with nhlify

## Next Steps

- **04_advanced_features.ipynb** - Learn about caching, rate limiting, and advanced transformations